In [ ]:
!pip install statsmodels openpyxl

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def calculate_fleiss_kappa_per_class(
    df_a1,
    df_a2,
    df_a3,
    id_col='full_text',
    label_col='label',
    valid_classes=None,
    label_mapping=None
):
    # Copy data
    a1 = df_a1[[id_col, label_col]].copy()
    a2 = df_a2[[id_col, label_col]].copy()
    a3 = df_a3[[id_col, label_col]].copy()

    # Normalisasi label
    for df in [a1, a2, a3]:
        df[label_col] = (
            df[label_col]
            .astype(str)
            .str.strip()
        )

    # Mapping label yang berbeda penulisan
    if label_mapping is not None:
        a1[label_col] = a1[label_col].replace(label_mapping)
        a2[label_col] = a2[label_col].replace(label_mapping)
        a3[label_col] = a3[label_col].replace(label_mapping)

    # Jika valid_classes diberikan,
    # hapus tweet yang memiliki kategori di luar kategori valid
    if valid_classes is not None:

        invalid_ids = set()

        for df in [a1, a2, a3]:
            invalid_ids.update(
                df.loc[
                    ~df[label_col].isin(valid_classes),
                    id_col
                ]
            )

        # Hapus tweet tersebut dari semua annotator
        a1 = a1[~a1[id_col].isin(invalid_ids)].copy()
        a2 = a2[~a2[id_col].isin(invalid_ids)].copy()
        a3 = a3[~a3[id_col].isin(invalid_ids)].copy()

    # Rename label
    a1 = a1.rename(columns={label_col: 'A1'})
    a2 = a2.rename(columns={label_col: 'A2'})
    a3 = a3.rename(columns={label_col: 'A3'})

     # Hapus duplikat tweet
    for df in [a1, a2, a3]:
        df.drop_duplicates(subset="full_text", keep="first", inplace=True)

    # Hapus missing value
    a1 = a1.dropna(subset=['A1'])
    a2 = a2.dropna(subset=['A2'])
    a3 = a3.dropna(subset=['A3'])

    # Merge berdasarkan tweet ID
    merged = (
        a1
        .merge(a2, on=id_col, how='inner')
        .merge(a3, on=id_col, how='inner')
    )

    print(f"Tweets with 3 annotations: {len(merged)}")

    # Daftar kelas
    if valid_classes is not None:
        classes = valid_classes
    else:
        classes = pd.unique(
            merged[['A1', 'A2', 'A3']].values.ravel()
        )

    results = []

    for cls in classes:

        # One-vs-rest
        binary = np.array([
            [
                int(row.A1 == cls),
                int(row.A2 == cls),
                int(row.A3 == cls)
            ]
            for row in merged.itertuples()
        ])

        # Jumlah annotator yang tidak memilih / memilih kelas
        table = np.column_stack([
            (binary == 0).sum(axis=1),
            (binary == 1).sum(axis=1)
        ])

        kappa = fleiss_kappa(
            table,
            method='fleiss'
        )

        results.append({
            'Class': cls,
            'Fleiss_Kappa': kappa
        })

    return pd.DataFrame(results), merged

In [ ]:
# Round 1

round1_a1 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_100_150_1.xlsx')
round1_a2 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_100_150_2.xlsx')
round1_a3 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_100_150_3.xlsx')

round1_kappa, round1_merged = calculate_fleiss_kappa_per_class(
    round1_a1,
    round1_a2,
    round1_a3,
    valid_classes=[
        'Kepadatan',
        'Keamanan',
        'Keterlambatan',
        'Pelayanan',
        'Kebersihan'
    ],
    label_mapping={
        'Kepadatan Penumpang': 'Kepadatan',
        'Pelayanan Petugas': 'Pelayanan'
    }
)

round1_kappa

Tweets with 3 annotations: 87


,Class,Fleiss_Kappa
0,Kepadatan,0.290344
1,Keamanan,0.400765
2,Keterlambatan,0.314835
3,Pelayanan,0.454355
4,Kebersihan,0.563253


In [ ]:
# Round 2

round2_a1 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_2_1.xlsx')
round2_a2 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_2_2.xlsx')
round2_a3 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_2_3.xlsx')

round2_kappa, round2_merged = calculate_fleiss_kappa_per_class(
    round2_a1,
    round2_a2,
    round2_a3,
    valid_classes=[
        'Kepadatan',
        'Keamanan',
        'Keterlambatan',
        'Pelayanan',
        'Kebersihan'
    ],
    label_mapping={
        'Kepadatan Penumpang': 'Kepadatan',
        'Pelayanan Petugas': 'Pelayanan'
    }
)

round2_kappa

Tweets with 3 annotations: 56


,Class,Fleiss_Kappa
0,Kepadatan,0.525256
1,Keamanan,0.183673
2,Keterlambatan,0.714815
3,Pelayanan,0.675188
4,Kebersihan,0.132660


In [ ]:
# Round 2

round3_a1 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_3_1.xlsx')
round3_a2 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_3_2.xlsx')
round3_a3 = pd.read_excel('/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_3_3.xlsx')

round3_kappa, round3_merged = calculate_fleiss_kappa_per_class(
    round3_a1,
    round3_a2,
    round3_a3,
    valid_classes=[
        'Kepadatan',
        'Keamanan',
        'Keterlambatan',
        'Pelayanan',
        'Kebersihan'
    ],
    label_mapping={
        'Kepadatan Penumpang': 'Kepadatan',
        'Pelayanan Petugas': 'Pelayanan'
    }
)

round3_kappa

Tweets with 3 annotations: 90


,Class,Fleiss_Kappa
0,Kepadatan,0.848485
1,Keamanan,0.908537
2,Keterlambatan,0.887874
3,Pelayanan,0.890208
4,Kebersihan,1.000000
